# Mangaka : entraînement sur GPU Kaggle

Ce notebook clone le dépôt, télécharge le dataset AnimeGAN, puis lance les scripts du dépôt
(rien n'est réécrit ici, tout le code vient de `src/`).

**Avant de lancer** : *Settings* → *Accelerator* : **GPU T4 x2** (ou P100), et *Internet* : **On**.
Remplir `REPO_URL` (et `BRANCH`) dans la cellule suivante.

À la fin, télécharger `/kaggle/working/resultats.zip` (checkpoints, ONNX, metrics.json, base MLflow)
et le décompresser à la racine du dépôt sur le PC, puis faire `dvc add` (voir README).

In [ ]:
REPO_URL = "https://github.com/<compte>/<depot>.git"  # à remplir
BRANCH = "develop"
STYLES = ["Hayao", "Shinkai"]  # modèle avancé : un générateur par style
EPOCHS_ANIMEGAN = 30
EPOCHS_CYCLEGAN = 20

In [ ]:
!nvidia-smi -L
!git clone --branch $BRANCH $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements-train.txt

## Données : dataset officiel AnimeGAN (release GitHub, 793 Mo)

In [ ]:
!wget -q https://github.com/TachibanaYoshino/AnimeGAN/releases/download/dataset-1/dataset.zip -O /kaggle/working/dataset.zip
!mkdir -p data/raw && unzip -q /kaggle/working/dataset.zip -d /kaggle/working/dataset_unzip
!ls /kaggle/working/dataset_unzip/*

In [ ]:
# Recopie vers l'arborescence attendue par src/data/dataset.py (voir data/README.md),
# puis contrôle qu'aucune photo de test n'est aussi dans les photos d'entraînement.
!python -m src.data.prepare_raw --src /kaggle/working/dataset_unzip --dst data/raw

In [ ]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////kaggle/working/repo/mlflow.db"
for style in STYLES:
    !python -m src.data.dataset --style {style}

## Baseline CycleGAN (128×128, Hayao)

In [ ]:
!python -m src.models.train_cyclegan --style Hayao --epochs $EPOCHS_CYCLEGAN --workers 2
!python -m src.models.evaluate --arch cyclegan --size 128 --style Hayao --checkpoint checkpoints/cyclegan-hayao/generator.pt

## Modèle avancé AnimeGAN (256×256), un générateur par style

Shinkai repart des poids Hayao (`--resume`) pour économiser du temps GPU.

In [ ]:
!python -m src.models.train --style Hayao --epochs $EPOCHS_ANIMEGAN
!python -m src.models.evaluate --style Hayao --checkpoint checkpoints/hayao/generator.pt --register

In [ ]:
if "Shinkai" in STYLES:
    !python -m src.models.train --style Shinkai --resume checkpoints/hayao/generator.pt --init-epochs 0 --epochs $EPOCHS_ANIMEGAN
    !python -m src.models.evaluate --style Shinkai --checkpoint checkpoints/shinkai/generator.pt --register

In [ ]:
!cat models/metrics.json
!cd /kaggle/working/repo && zip -qr /kaggle/working/resultats.zip checkpoints models mlflow.db mlruns
!ls -lh /kaggle/working/resultats.zip